In [1]:
import xarray as xr
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import easysnowdata
import numpy as np
import rasterio
import os
import contextily as ctx
import shutil


DATA_DIR = Path("data/NorSWE")
NORSWE_ZARR_FILEPATH = DATA_DIR / "NorSWE.zarr"

MAX_NORSWE_TIMING_OUTPUT_ZARR_FILEPATH = Path('data/comparison_datasets/max_NorSWE_swe_timing.zarr')
RUNOFF_ONSET_STATION_CHIPS_OUTPUT_ZARR_FILEPATH = Path('data/comparison_datasets/runoff_onset_NorSWE_station_chips.zarr')

In [2]:
NorSWE_ds = xr.open_dataset(NORSWE_ZARR_FILEPATH)
NorSWE_ds

<xarray.Dataset> Size: 3GB
Dimensions:        (time: 15706, station_id: 10153)
Coordinates:
  * time           (time) datetime64[ns] 126kB 1979-01-01 ... 2021-12-31
  * station_id     (station_id) <U32 1MB 'CanSWE-ALE-05AA805' ... 'NVE-2.277'
    elevation      (station_id) float32 41kB ...
    latitude       (station_id) float32 41kB ...
    longitude      (station_id) float32 41kB ...
    mmask          (station_id) uint8 10kB ...
    source         (station_id) <U63 3MB ...
    station_name   (station_id) <U63 3MB ...
    type_mes       (station_id) uint8 10kB ...
Data variables:
    data_flag_snd  (time, station_id) int8 159MB ...
    data_flag_snw  (time, station_id) int8 159MB ...
    den            (time, station_id) float32 638MB ...
    qc_flag_snd    (time, station_id) int8 159MB ...
    qc_flag_snw    (time, station_id) int8 159MB ...
    snd            (time, station_id) float32 638MB ...
    snw            (time, station_id) float32 638MB ...
Attributes:
    Conventions:    CF-1.9
    Title:          Northern Hemisphere in situ SWE 1979-2021 v3
    Latest_update:  April 2025
    Source:         Environment and Climate Change Canada and partners (https...
    Distribution:   CanSWE data are redistributed under the Open Government L...
    Comment:        See Vionnet et al. (ESSD, 2021) for a description of the ...
    source_url:     https://zenodo.org/records/15263370
    reference:      Pirazzini et al., 2025, ESSD, https://essd.copernicus.org...

In [3]:
NorSWE_ds.coords['water_year'] = ("time", pd.to_datetime(NorSWE_ds['time']).map(easysnowdata.utils.datetime_to_WY))
NorSWE_ds.coords['DOWY'] = ("time", pd.to_datetime(NorSWE_ds['time']).map(easysnowdata.utils.datetime_to_DOWY))
NorSWE_ds

<xarray.Dataset> Size: 3GB
Dimensions:        (time: 15706, station_id: 10153)
Coordinates:
  * time           (time) datetime64[ns] 126kB 1979-01-01 ... 2021-12-31
    water_year     (time) int64 126kB 1979 1979 1979 1979 ... 2022 2022 2022
    DOWY           (time) int64 126kB 93 94 95 96 97 98 99 ... 87 88 89 90 91 92
  * station_id     (station_id) <U32 1MB 'CanSWE-ALE-05AA805' ... 'NVE-2.277'
    elevation      (station_id) float32 41kB ...
    latitude       (station_id) float32 41kB ...
    longitude      (station_id) float32 41kB ...
    mmask          (station_id) uint8 10kB ...
    source         (station_id) <U63 3MB ...
    station_name   (station_id) <U63 3MB ...
    type_mes       (station_id) uint8 10kB ...
Data variables:
    data_flag_snd  (time, station_id) int8 159MB ...
    data_flag_snw  (time, station_id) int8 159MB ...
    den            (time, station_id) float32 638MB ...
    qc_flag_snd    (time, station_id) int8 159MB ...
    qc_flag_snw    (time, station_id) int8 159MB ...
    snd            (time, station_id) float32 638MB ...
    snw            (time, station_id) float32 638MB ...
Attributes:
    Conventions:    CF-1.9
    Title:          Northern Hemisphere in situ SWE 1979-2021 v3
    Latest_update:  April 2025
    Source:         Environment and Climate Change Canada and partners (https...
    Distribution:   CanSWE data are redistributed under the Open Government L...
    Comment:        See Vionnet et al. (ESSD, 2021) for a description of the ...
    source_url:     https://zenodo.org/records/15263370
    reference:      Pirazzini et al., 2025, ESSD, https://essd.copernicus.org...

In [4]:
# filter just to water years 2015-2024, use coordinate WY
NorSWE_ds = NorSWE_ds.sel(time=NorSWE_ds['water_year'].isin([2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]))
#remove WY 2022 because it is incomplete
NorSWE_ds = NorSWE_ds.sel(time=~NorSWE_ds['water_year'].isin([2022]))
NorSWE_ds

<xarray.Dataset> Size: 422MB
Dimensions:        (time: 2557, station_id: 10153)
Coordinates:
  * time           (time) datetime64[ns] 20kB 2014-10-01 ... 2021-09-30
    water_year     (time) int64 20kB 2015 2015 2015 2015 ... 2021 2021 2021 2021
    DOWY           (time) int64 20kB 1 2 3 4 5 6 7 ... 360 361 362 363 364 365
  * station_id     (station_id) <U32 1MB 'CanSWE-ALE-05AA805' ... 'NVE-2.277'
    elevation      (station_id) float32 41kB ...
    latitude       (station_id) float32 41kB ...
    longitude      (station_id) float32 41kB ...
    mmask          (station_id) uint8 10kB ...
    source         (station_id) <U63 3MB ...
    station_name   (station_id) <U63 3MB ...
    type_mes       (station_id) uint8 10kB ...
Data variables:
    data_flag_snd  (time, station_id) int8 26MB ...
    data_flag_snw  (time, station_id) int8 26MB ...
    den            (time, station_id) float32 104MB ...
    qc_flag_snd    (time, station_id) int8 26MB ...
    qc_flag_snw    (time, station_id) int8 26MB ...
    snd            (time, station_id) float32 104MB ...
    snw            (time, station_id) float32 104MB ...
Attributes:
    Conventions:    CF-1.9
    Title:          Northern Hemisphere in situ SWE 1979-2021 v3
    Latest_update:  April 2025
    Source:         Environment and Climate Change Canada and partners (https...
    Distribution:   CanSWE data are redistributed under the Open Government L...
    Comment:        See Vionnet et al. (ESSD, 2021) for a description of the ...
    source_url:     https://zenodo.org/records/15263370
    reference:      Pirazzini et al., 2025, ESSD, https://essd.copernicus.org...

In [5]:
measurement_type_value_counts = NorSWE_ds['type_mes'].to_pandas().value_counts()
measurement_type_value_counts.index = measurement_type_value_counts.index.map({0: 'multi point manual snow survey', 1: 'Single-point manual snow water equivalent measurement', 2: 'snow pillow or snow scale', 3: 'passive gamma (automated)', 64: 'airborne passive gamma'})
measurement_type_value_counts

type_mes
multi point manual snow survey                           6672
airborne passive gamma                                   2357
snow pillow or snow scale                                1000
passive gamma (automated)                                 113
Single-point manual snow water equivalent measurement      11
Name: count, dtype: int64

In [6]:
NorSWE_swe_da = NorSWE_ds['snw'] #.where(NorSWE_ds['data_flag_snw']) .where(NorSWE_ds['qc_flag_snw'])
NorSWE_swe_da

<xarray.DataArray 'snw' (time: 2557, station_id: 10153)> Size: 104MB
[25961221 values with dtype=float32]
Coordinates:
  * time          (time) datetime64[ns] 20kB 2014-10-01 ... 2021-09-30
    water_year    (time) int64 20kB 2015 2015 2015 2015 ... 2021 2021 2021 2021
    DOWY          (time) int64 20kB 1 2 3 4 5 6 7 ... 360 361 362 363 364 365
  * station_id    (station_id) <U32 1MB 'CanSWE-ALE-05AA805' ... 'NVE-2.277'
    elevation     (station_id) float32 41kB ...
    latitude      (station_id) float32 41kB ...
    longitude     (station_id) float32 41kB ...
    mmask         (station_id) uint8 10kB ...
    source        (station_id) <U63 3MB ...
    station_name  (station_id) <U63 3MB ...
    type_mes      (station_id) uint8 10kB 0 0 0 0 0 0 0 0 0 ... 2 2 0 0 0 0 0 0
Attributes:
    standard_name:  surface_snow_amount
    long_name:      Surface snow water equivalent
    units:          kg m**-2

In [7]:
max_swe_timing_ds = NorSWE_swe_da.groupby("water_year").max().to_dataset(name='max_SWE_value')
max_swe_timing_ds

<xarray.Dataset> Size: 7MB
Dimensions:        (station_id: 10153, water_year: 7)
Coordinates:
  * station_id     (station_id) <U32 1MB 'CanSWE-ALE-05AA805' ... 'NVE-2.277'
    elevation      (station_id) float32 41kB ...
    latitude       (station_id) float32 41kB ...
    longitude      (station_id) float32 41kB ...
    mmask          (station_id) uint8 10kB ...
    source         (station_id) <U63 3MB ...
    station_name   (station_id) <U63 3MB ...
    type_mes       (station_id) uint8 10kB 0 0 0 0 0 0 0 0 0 ... 2 2 0 0 0 0 0 0
  * water_year     (water_year) int64 56B 2015 2016 2017 2018 2019 2020 2021
Data variables:
    max_SWE_value  (water_year, station_id) float32 284kB nan nan ... nan 473.0

In [8]:
def find_pct_max_timing(da, pct, dim='time', skipna=True):
    """Find the time when SWE last crosses below a percentage of max SWE"""
    max_val = da.max(dim=dim, skipna=skipna)
    threshold = max_val * pct
    # Create boolean mask of values above threshold
    above_thresh = xr.where(da >= threshold, 1, np.nan)
    # Find the last True value
    return above_thresh.sel(time=slice(None, None, -1)).swap_dims({'time':'DOWY'}).idxmax(dim="DOWY", skipna=True).where(lambda x: x>0) 

In [9]:
pct_list = [1.0, 0.99, 0.95, 0.9, 0.5]
for pct in pct_list:
    pct_str = str(int(pct * 100))
    max_swe_timing_ds[f'{pct_str}pct_of_max_SWE_timing'] = NorSWE_swe_da.groupby("water_year").map(lambda x: find_pct_max_timing(x, pct)).where(lambda x: x>0)

max_swe_timing_ds

<xarray.Dataset> Size: 10MB
Dimensions:                   (station_id: 10153, water_year: 7)
Coordinates:
  * station_id                (station_id) <U32 1MB 'CanSWE-ALE-05AA805' ... ...
    elevation                 (station_id) float32 41kB ...
    latitude                  (station_id) float32 41kB ...
    longitude                 (station_id) float32 41kB ...
    mmask                     (station_id) uint8 10kB ...
    source                    (station_id) <U63 3MB ...
    station_name              (station_id) <U63 3MB ...
    type_mes                  (station_id) uint8 10kB 0 0 0 0 0 0 ... 0 0 0 0 0
  * water_year                (water_year) int64 56B 2015 2016 ... 2020 2021
Data variables:
    max_SWE_value             (water_year, station_id) float32 284kB nan ... ...
    100pct_of_max_SWE_timing  (water_year, station_id) float64 569kB nan ... ...
    99pct_of_max_SWE_timing   (water_year, station_id) float64 569kB nan ... ...
    95pct_of_max_SWE_timing   (water_year, station_id) float64 569kB nan ... ...
    90pct_of_max_SWE_timing   (water_year, station_id) float64 569kB nan ... ...
    50pct_of_max_SWE_timing   (water_year, station_id) float64 569kB nan ... ...

In [10]:
MAX_NORSWE_TIMING_OUTPUT_ZARR_FILEPATH.parent.mkdir(parents=True, exist_ok=True)
max_swe_timing_ds.to_zarr(MAX_NORSWE_TIMING_OUTPUT_ZARR_FILEPATH, mode='w')
print(f"Written -> {MAX_NORSWE_TIMING_OUTPUT_ZARR_FILEPATH}")

Written -> data/comparison_datasets/max_NorSWE_swe_timing.zarr


/home/eric/repos/global_snowmelt_runoff_onset/.pixi/envs/default/lib/python3.14/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=32, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/home/eric/repos/global_snowmelt_runoff_onset/.pixi/envs/default/lib/python3.14/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=63, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with th

In [11]:
NorSWE_gdf = gpd.GeoDataFrame(
    {
        'station_id':   NorSWE_ds['station_id'].values,
        'station_name': NorSWE_ds['station_name'].values,
        'source':       NorSWE_ds['source'].values,
        'elevation':    NorSWE_ds['elevation'].values,
        'type_mes':     NorSWE_ds['type_mes'].values,
        'mmask':        NorSWE_ds['mmask'].values,
    },
    geometry=gpd.points_from_xy(NorSWE_ds['longitude'].values, NorSWE_ds['latitude'].values),
    crs='EPSG:4326',
)

NorSWE_gdf

,station_id,station_name,source,elevation,type_mes,mmask,geometry
0,CanSWE-ALE-05AA805,WEST CASTLE SNOW,CanSWEv6-Alberta Environment,1525.000000,0,1,POINT (-114.35 49.26667)
1,CanSWE-ALE-05AA806,RACE HORSE CREEK,CanSWEv6-Alberta Environment,1920.000000,0,1,POINT (-114.63333 49.81667)
2,CanSWE-ALE-05AD802,MIDDLE DRYWOOD,CanSWEv6-Alberta Environment,1570.000000,0,1,POINT (-114.05 49.25)
3,CanSWE-ALE-05AE810,\LEE CREEK \\P\\\,CanSWEv6-Alberta Environment,1525.000000,0,0,POINT (-113.56667 49.03333)
4,CanSWE-ALE-05AE811,\LEE CREEK \\D\\\,CanSWEv6-Alberta Environment,1660.000000,0,1,POINT (-113.61667 49.01667)
...,...,...,...,...,...,...,...
10148,NVE-2.286,ATNA-900,NVE,900.000000,0,1,POINT (10.04563 61.96072)
10149,NVE-2.293,ATNA-1000,NVE,1000.000000,0,1,POINT (10.05904 61.95533)
10150,NVE-2.308,ATNA-1100,NVE,1100.000000,0,1,POINT (10.07666 61.94841)
10151,NVE-2.309,ATNA-1200,NVE,1200.000000,0,1,POINT (10.09257 61.95081)


In [12]:
NorSWE_gdf.explore(
    column='elevation',
    cmap='terrain',
    tooltip=['station_id', 'station_name', 'source', 'elevation'],
    marker_kwds={'radius': 4},
)

## Now create a dataset of NorSWE runoff onset chips, first test on small area

In [13]:
# now bring in the runoff onset data......
from global_snowmelt_runoff_onset.config import Config
config = Config('config/global_config_v9.txt')

SAS token is valid until 2026-07-03 00:51 UTC (406.7 hours)
----------------------------------------
Configuration loaded:
config_name = global_config_v9
version = v9
resolution = 0.00072000072000072
bands = vv
mountain_snow_only = False
spatial_chunk_dim_s1_read = 2048
spatial_chunk_dim_s1_process = 512
spatial_chunk_dim_zarr_output = 2048
bbox_left = -179.999
bbox_right = 179.999
bbox_top = 81.099
bbox_bottom = -59.999
wy_start = 2015
wy_end = 2024
low_backscatter_threshold = 0.001
min_monthly_acquisitions = 1
max_allowed_days_gap_per_orbit = 30
min_years_for_median_std = 3
extend_search_window_beyond_sdd_days = 16
min_consec_snow_days_for_seasonal_snow = 56
valid_tiles_geojson_path = processing/tile_data/global_tiles_with_seasonal_snow.geojson
tile_results_path = processing/tile_data/tile_results_v9.csv
global_runoff_zarr_store_azure_path = snowmelt/snowmelt_runoff_onset/global_v9.zarr
seasonal_snow_mask_zarr_store_azure_path = snowmelt/snow_cover/global_modis_snow_cover.zarr
season

In [14]:
easysnowdata.remote_sensing.get_seasonal_snow_classification??

Signature:
easysnowdata.remote_sensing.get_seasonal_snow_classification(
    bbox_input: geopandas.geodataframe.GeoDataFrame | tuple | shapely.geometry.base.BaseGeometry | None = None,
    mask_nodata: bool = False,
) -> xarray.core.dataarray.DataArray
Source:   
def get_seasonal_snow_classification(bbox_input: gpd.GeoDataFrame | tuple | shapely.geometry.base.BaseGeometry | None = None, mask_nodata: bool = False,
) -> xr.DataArray:
    """
    Fetches 10arcsec (~300m) Sturm & Liston 2021 seasonal snow classification data for a given bounding box.

    Description:
    This dataset consists of global, seasonal snow classifications determined from air temperature,
    precipitation, and wind speed climatologies. This is the 10 arcsec (~300m) product in EPSG:4326.

    Parameters
    ----------
    bbox_input : geopandas.GeoDataFrame or tuple or Shapely Geometry
        GeoDataFrame containing the bounding box, or a tuple of (xmin, ymin, xmax, ymax), or a Shapely geometry.
    mask_nodata

In [ ]:
runoff_onset_ds = xr.open_zarr(config.global_runoff_store, consolidated=True,decode_coords='all')
runoff_onset_ds

In [ ]:
def get_station_gdf(NorSWE_gdf, station_id, buffer_radius=None):
    station_gdf = NorSWE_gdf[NorSWE_gdf.station_id==station_id]
    station_epsg = station_gdf.estimate_utm_crs().to_epsg()
    station_gdf = station_gdf.to_crs(epsg=station_epsg)
    if buffer_radius:
        station_gdf['geometry'] = station_gdf.geometry.buffer(buffer_radius)
    return station_gdf

In [ ]:
station_id = 'SNOTEL-679'
max_buffer_radius = 1000
station_utm_gdf = get_station_gdf(NorSWE_gdf, station_id, buffer_radius=max_buffer_radius)
station_utm_gdf

In [ ]:
import rioxarray as rxr
snow_classification_da = rxr.open_rasterio(
    "https://uwcryo.blob.core.windows.net/snowmelt/eric/snow_classification/SnowClass_GL_300m_10.0arcsec_2021_v01.0.tif",
    chunks=True,
    mask_and_scale=True,
).squeeze()
snow_classification_da

In [ ]:
def get_station_buffered_runoff_onset(runoff_onset_ds, station_utm_gdf):

    runoff_onset_station_ds = runoff_onset_ds.rio.clip_box(*station_utm_gdf.total_bounds, crs=station_utm_gdf.crs).rio.reproject(station_utm_gdf.crs).astype('float32')
    
    runoff_onset_station_ds['fcf'] = easysnowdata.remote_sensing.get_forest_cover_fraction(runoff_onset_station_ds.rio.transform_bounds('EPSG:4326'),mask_nodata=True).rio.reproject_match(runoff_onset_station_ds,resampling=rasterio.enums.Resampling.bilinear)
    runoff_onset_station_ds['dem'] = easysnowdata.topography.get_copernicus_dem(runoff_onset_station_ds.rio.transform_bounds('EPSG:4326'), resolution=30).rio.reproject_match(runoff_onset_station_ds,resampling=rasterio.enums.Resampling.bilinear)
    runoff_onset_station_ds['worldcover'] = easysnowdata.remote_sensing.get_esa_worldcover(runoff_onset_station_ds.rio.transform_bounds('EPSG:4326'),mask_nodata=True).rio.reproject_match(runoff_onset_station_ds,resampling=rasterio.enums.Resampling.nearest)
    runoff_onset_station_ds['snow_class'] = snow_classification_da.rio.clip_box(*station_utm_gdf.total_bounds, crs=station_utm_gdf.crs).rio.reproject_match(runoff_onset_station_ds,resampling=rasterio.enums.Resampling.nearest)
    # what if we added Sturm and Liston snow classification?

    runoff_onset_station_ds = runoff_onset_station_ds.rio.clip(station_utm_gdf.geometry) # ,all_touched=True

    return runoff_onset_station_ds

In [ ]:
runoff_onset_station_ds = get_station_buffered_runoff_onset(runoff_onset_ds, station_utm_gdf)
runoff_onset_station_ds

In [ ]:
runoff_onset_station_ds['runoff_onset'].plot.imshow(col='water_year', col_wrap=5)

In [ ]:
runoff_onset_station_ds['temporal_resolution'].plot.imshow(col='water_year', col_wrap=5)

In [ ]:
runoff_onset_station_ds['fcf'].example_plot(runoff_onset_station_ds['fcf'])

In [ ]:
runoff_onset_station_ds['worldcover'].example_plot(runoff_onset_station_ds['worldcover'])

In [ ]:
runoff_onset_station_ds['snow_class'].plot.imshow()

In [ ]:
os.environ['PLANET_API_KEY'] = 'PLAKc3df9ed9f45f40a79cd9d9bebee3bb50'

def get_planet_basemap_from_gdf(gdf, mosaic_name="global_monthly_2025_08_mosaic", zoom="auto"):
    api_key = os.environ["PLANET_API_KEY"]
    url = f"https://tiles.planet.com/basemaps/v1/planet-tiles/{mosaic_name}/gmap/{{z}}/{{x}}/{{y}}.png?api_key={api_key}"

    gdf_3857 = gdf.to_crs(epsg=3857)

    img, extent = ctx.bounds2img(
        *gdf_3857.total_bounds,
        zoom=zoom,
        source=url,
        ll=False,
    )

    left, right, bottom, top = extent
    h, w = img.shape[:2]

    da = xr.DataArray(
        np.moveaxis(img[:, :, :3], -1, 0).astype(np.float32),
        dims=["band", "y", "x"],
        coords={
            "band": [1, 2, 3],
            "y": np.linspace(top, bottom, h),
            "x": np.linspace(left, right, w),
        },
    )
    da = da.rio.write_crs("EPSG:3857")
    da = da.rio.reproject(gdf.crs)
    da = da.rio.clip(gdf.to_crs(gdf.crs).geometry, all_touched=True)
    da.attrs["mosaic"] = mosaic_name

    return da

In [ ]:
planet_da = get_planet_basemap_from_gdf(station_utm_gdf)
planet_da

In [ ]:
f,ax=plt.subplots(figsize=(10,10))
planet_da.plot.imshow(ax=ax,robust=True)
ax.set_aspect('equal')
ax.set_title(f"Planet {planet_da.attrs['mosaic']}")

## Build per-station chip Zarr with relative coordinates

For each NorSWE station, extract a 25×25 pixel spatial chip (1000 m buffer, 80 m pixels) from the global SAR runoff onset Zarr. Instead of absolute UTM x/y coordinates — which differ per station and can't be stacked — store offsets in **meters from the station center** (`x_rel`, `y_rel`). Every chip then has identical coordinate arrays and can be concatenated along a `station_id` dimension in a single Zarr.

The loop is **resumable**: it checks which stations are already in the Zarr at startup and skips them.

Output: `data/comparison_datasets/runoff_onset_and_max_swe_timing_station_chips.zarr`

In [ ]:
from tqdm.auto import tqdm

# --- chip geometry ---
BUFFER_RADIUS = 1000   # meters — defines the chip footprint
PIXEL_SIZE    = 80     # meters — native SAR resolution
N_HALF        = 12     # pixels each side; chip size = 2*N_HALF+1 = 25
TARGET_REL    = np.arange(-N_HALF, N_HALF + 1) * PIXEL_SIZE  # [-960, -880, ..., 0, ..., 960] m

# --- temporal overlap of SAR dataset (WY 2015-2024) and NorSWE (WY 2015-2021) ---
TARGET_WATER_YEARS = [2015, 2016, 2017, 2018, 2019, 2020, 2021]

BATCH_SIZE  = 30

# Max station_id string length — used to normalise dtype across batches so Zarr
# doesn't raise "Mismatched dtypes" when appending (each batch may have a different
# longest ID, giving a different <UN dtype).
MAX_SID_LEN = max(len(s) for s in NorSWE_gdf['station_id'].values)

In [ ]:
def extract_chip_with_relative_coords(
    runoff_onset_ds, NorSWE_gdf, station_id,
    buffer_radius=BUFFER_RADIUS, pixel_size=PIXEL_SIZE, target_rel=TARGET_REL,
):
    """
    Extract a spatial chip from the global runoff onset Zarr around a NorSWE station.

    Absolute UTM x/y are replaced with offsets (meters from station center) so that
    every chip shares identical x_rel/y_rel coordinate arrays and can be concatenated
    along a station_id dimension.

    The station's absolute UTM center coordinates and EPSG code are stored as scalar
    coordinates (station_utm_x, station_utm_y, station_utm_epsg). The full per-pixel
    absolute UTM coordinate arrays are stored as data variables x_abs (dims: x_rel)
    and y_abs (dims: y_rel), preserving the true reprojected pixel centers.

    Steps
    -----
    1. Clip and reproject the global Zarr to local UTM at exactly PIXEL_SIZE metres.
    2. Fetch auxiliary layers (fcf, dem, worldcover) while the chip still carries
       absolute coordinates and a valid CRS (needed for reproject_match).
    3. Capture x_abs/y_abs as data variables (dims x/y) before converting coords;
       rename({'x': 'x_rel', 'y': 'y_rel'}) propagates the dim names automatically.
    4. Store the station UTM centre and EPSG as scalar coordinates.
    5. Convert x/y → x_rel/y_rel by subtracting the station centre.
    6. Snap to TARGET_REL via reindex so every chip has an identical coordinate grid.
    """
    # Buffer station in local UTM
    station_utm_gdf = get_station_gdf(NorSWE_gdf, station_id, buffer_radius=buffer_radius)
    station_x = float(station_utm_gdf.geometry.centroid.x.iloc[0])
    station_y = float(station_utm_gdf.geometry.centroid.y.iloc[0])
    station_crs = station_utm_gdf.crs

    # Expand clip box by one pixel on each side to guarantee edge pixels survive reindex
    pad = pixel_size
    minx, miny, maxx, maxy = station_utm_gdf.total_bounds
    padded_bounds = (minx - pad, miny - pad, maxx + pad, maxy + pad)

    # Clip from global Zarr and reproject to local UTM at a fixed pixel size
    chip = (
        runoff_onset_ds[['runoff_onset', 'temporal_resolution']]
        .rio.clip_box(*padded_bounds, crs=station_crs)
        .rio.reproject(station_crs, resolution=pixel_size)
        .sel(water_year=TARGET_WATER_YEARS)
        .astype('float32')
    )

    # Fetch auxiliary layers while the chip still has absolute coords + CRS
    bounds_4326 = chip.rio.transform_bounds('EPSG:4326')
    chip['fcf'] = (
        easysnowdata.remote_sensing
        .get_forest_cover_fraction(bounds_4326, mask_nodata=True)
        .rio.reproject_match(chip, resampling=rasterio.enums.Resampling.bilinear)
    )
    chip['dem'] = (
        easysnowdata.topography
        .get_copernicus_dem(bounds_4326, resolution=30)
        .rio.reproject_match(chip, resampling=rasterio.enums.Resampling.bilinear)
    )
    chip['worldcover'] = (
        easysnowdata.remote_sensing
        .get_esa_worldcover(bounds_4326, mask_nodata=True)
        .rio.reproject_match(chip, resampling=rasterio.enums.Resampling.nearest)
    )

    # Preserve original absolute UTM pixel coords as data variables *before* rename.
    # rename({'x': 'x_rel', 'y': 'y_rel'}) propagates dims automatically so that
    # x_abs ends up with dim 'x_rel' and y_abs with dim 'y_rel' — no recalculation.
    chip['x_abs'] = xr.DataArray(chip.x.values, dims=['x'])
    chip['y_abs'] = xr.DataArray(chip.y.values, dims=['y'])

    # Record the absolute UTM centre as scalar coords
    chip = chip.assign_coords(
        station_utm_x    = station_x,
        station_utm_y    = station_y,
        station_utm_epsg = station_crs.to_epsg(),
    )

    # Convert absolute x/y → relative coordinates (metres from station centre)
    chip = (
        chip
        .assign_coords(x=chip.x.values - station_x, y=chip.y.values - station_y)
        .rename({'x': 'x_rel', 'y': 'y_rel'})
    )

    # Snap to the fixed target grid; any remaining edge gap fills with NaN.
    # x_abs/y_abs are reindexed the same way, preserving the true pixel centers.
    chip = chip.reindex(
        x_rel=target_rel, y_rel=target_rel,
        method='nearest', tolerance=pixel_size // 2,
    )

    chip = chip.drop_vars('spatial_ref', errors='ignore')

    # Strip all variable and dataset attrs — rioxarray/reproject_match can leave
    # non-JSON-serializable objects (e.g. function references) that break Zarr writes.
    for var in list(chip.data_vars) + list(chip.coords):
        chip[var].attrs = {}
    chip.attrs = {}

    return chip

In [ ]:
# Smoke test on SNOTEL-679 (Paradise, WA) — compare runoff_onset to existing cell 17
test_chip = extract_chip_with_relative_coords(runoff_onset_ds, NorSWE_gdf, 'SNOTEL-679')
test_chip

In [ ]:
assert list(test_chip.x_rel.values) == list(TARGET_REL), "x_rel mismatch"
assert list(test_chip.y_rel.values) == list(TARGET_REL), "y_rel mismatch"
test_chip['runoff_onset'].plot.imshow(col='water_year', col_wrap=4)

In [ ]:
# --- Resume check ---
# Run this cell at the start of each session to determine which stations remain.
RUNOFF_ONSET_STATION_CHIPS_OUTPUT_ZARR_FILEPATH.parent.mkdir(parents=True, exist_ok=True)

if RUNOFF_ONSET_STATION_CHIPS_OUTPUT_ZARR_FILEPATH.exists():
    try:
        _existing = xr.open_zarr(RUNOFF_ONSET_STATION_CHIPS_OUTPUT_ZARR_FILEPATH)
        already_done = set(_existing.station_id.values)
        _existing.close()
        zarr_initialized = True
        print(f"Resuming: {len(already_done)}/{len(NorSWE_gdf)} ({100*len(already_done)/len(NorSWE_gdf):.2f}%) stations already written")
        print(140*'-')
        print("Existing Zarr contents:")
        print(_existing)
    except Exception as e:
        print(f"Existing Zarr is corrupt ({e}); deleting and starting fresh.")
        shutil.rmtree(RUNOFF_ONSET_STATION_CHIPS_OUTPUT_ZARR_FILEPATH)
        already_done = set()
        zarr_initialized = False
else:
    already_done = set()
    zarr_initialized = False
    print("Starting fresh")

remaining_ids = [sid for sid in NorSWE_gdf['station_id'].values if sid not in already_done]
print(140*'-')
print(f"{len(remaining_ids)} stations remaining")

In [ ]:
import warnings
import rasterio.errors
from concurrent.futures import ThreadPoolExecutor, as_completed
import gc

N_WORKERS = 12   # I/O-bound; lower to 4 if rate-limit errors appear from Zenodo/Planetary Computer

chunk_spec = {
    'station_id': BATCH_SIZE,
    'water_year': len(TARGET_WATER_YEARS),
    'y_rel': len(TARGET_REL),
    'x_rel': len(TARGET_REL),
}

def _extract_one(sid):
    try:
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore', category=rasterio.errors.NotGeoreferencedWarning)
            chip = extract_chip_with_relative_coords(runoff_onset_ds, NorSWE_gdf, sid)

        chip = chip.expand_dims('station_id').assign_coords(station_id=[sid])

        # Bind scalar station coords to station_id dim so xr.concat always produces
        # ('station_id',) dims. Without this, xr.concat (coords='different') collapses
        # them to scalar () when all chips in a batch share the same UTM zone, which
        # mismatches the ('station_id',) dims already on disk → ValueError on append.
        chip = chip.assign_coords(
            station_utm_x    = ('station_id', [float(chip.station_utm_x)]),
            station_utm_y    = ('station_id', [float(chip.station_utm_y)]),
            station_utm_epsg = ('station_id', [int(chip.station_utm_epsg)]),
        )

        # Materialise into numpy — clears dask graph references so Azure read memory
        # is released immediately rather than accumulating across concurrent workers.
        chip = chip.load()
        return sid, chip, None
    except Exception as e:
        return sid, None, e

for batch_idx, batch_start in enumerate(
    tqdm(range(0, len(remaining_ids), BATCH_SIZE), desc="Batches")
):
    batch_ids = remaining_ids[batch_start : batch_start + BATCH_SIZE]
    batch_chips = []

    with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = {executor.submit(_extract_one, sid): sid for sid in batch_ids}
        for future in tqdm(as_completed(futures), total=len(futures), desc=f"  Batch {batch_idx}", leave=False):
            sid, chip, err = future.result()
            if err:
                print(f"  Skipped {sid}: {err}")
            else:
                batch_chips.append(chip)

    if not batch_chips:
        continue

    # coords='all' forces station_utm_x/y/epsg to be concatenated along station_id
    # even when all chips in a batch share the same UTM zone value. Without this,
    # xr.concat collapses identical coords to scalar (), mismatching dims on disk.
    batch_ds = xr.concat(batch_chips, dim='station_id', coords='all').chunk(chunk_spec)

    # Normalise station_id to a fixed-width string dtype so every batch matches
    # the first write. xarray infers <UN from the longest ID in each batch, which
    # varies, causing Zarr to raise "Mismatched dtypes" on append.
    batch_ds = batch_ds.assign_coords(
        station_id=batch_ds.station_id.values.astype(f'U{MAX_SID_LEN}')
    )

    if not zarr_initialized:
        batch_ds.to_zarr(RUNOFF_ONSET_STATION_CHIPS_OUTPUT_ZARR_FILEPATH, mode='w')
        zarr_initialized = True
    else:
        batch_ds.to_zarr(RUNOFF_ONSET_STATION_CHIPS_OUTPUT_ZARR_FILEPATH, append_dim='station_id')

    del batch_chips, batch_ds
    gc.collect()

print(f"Done — {RUNOFF_ONSET_STATION_CHIPS_OUTPUT_ZARR_FILEPATH}")

In [ ]:
# Verification
chips_ds = xr.open_zarr(RUNOFF_ONSET_STATION_CHIPS_OUTPUT_ZARR_FILEPATH)
print(chips_ds)

assert np.array_equal(chips_ds.x_rel.values, TARGET_REL), "x_rel mismatch"
assert np.array_equal(chips_ds.y_rel.values, TARGET_REL), "y_rel mismatch"
print(f"\nx_rel/y_rel identical across all stations ✓  ({len(TARGET_REL)} values: {TARGET_REL[0]}…{TARGET_REL[-1]} m)")

# Spot-check SNOTEL-679 against the smoke-test chip
chips_ds['runoff_onset'].sel(station_id='SNOTEL-679').plot.imshow(col='water_year', col_wrap=4)

In [ ]:
station_with_nodata = "CanSWE-SCD-NT002"

In [ ]:
max_buffer_radius = 1000
station_nodata_utm_gdf = get_station_gdf(NorSWE_gdf, station_with_nodata, buffer_radius=max_buffer_radius)
station_nodata_utm_gdf

In [ ]:
f,ax=plt.subplots(figsize=(10,10))
station_nodata_utm_gdf.plot(ax=ax, edgecolor='red', facecolor='none', linewidth=2)
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery,crs=station_nodata_utm_gdf.crs)
ax.set_aspect('equal')
ax.set_title(f"Station with no data: {station_with_nodata}")

# add an inset axis to show the a orthographic zoom out
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
inset_ax = inset_axes(ax, width="40%", height="40%", loc='upper right')
inset_ax.set_aspect('equal')
station_nodata_utm_gdf.plot(ax=inset_ax, edgecolor='red', facecolor='none', linewidth=1)
inset_ax.set_xlim(station_nodata_utm_gdf.total_bounds[[0, 2]] + np.array([-max_buffer_radius*200, max_buffer_radius*200]))
inset_ax.set_ylim(station_nodata_utm_gdf.total_bounds[[1, 3]] + np.array([-max_buffer_radius*200, max_buffer_radius*200]))
ctx.add_basemap(inset_ax, source=ctx.providers.Esri.WorldImagery,crs=station_nodata_utm_gdf.crs)
inset_ax.set_title("Zoomed out")

## Combine chip Zarr and SWE timing Zarr

After both Zarrs are complete, merge them in memory (or write a combined Zarr).
`max_swe_timing_ds` has dims `(station_id, water_year)` and shares the `station_id` dimension with the chip dataset, so `xr.merge` aligns on both.

In [ ]:
runoff_onset_chips_ds = xr.open_zarr(RUNOFF_ONSET_STATION_CHIPS_OUTPUT_ZARR_FILEPATH)
runoff_onset_chips_ds

In [ ]:
max_swe_timing_ds = xr.open_zarr(MAX_NORSWE_TIMING_OUTPUT_ZARR_FILEPATH)
max_swe_timing_ds

In [ ]:
NorSWE_runoff_onset_chips_and_max_swe_timing_ds = xr.merge([runoff_onset_chips_ds, max_swe_timing_ds], join='inner')
NorSWE_runoff_onset_chips_and_max_swe_timing_ds